# Segundo rotulador (Qwen3.5-9B) e baselines Qwen3.5 no teste

Projeto classificador-reclamacoes. Este notebook:
1. instala o Ollama e baixa `qwen3.5:9b` e `qwen3.5:2b` (os mesmos modelos do Ollama local);
2. rotula teste + 1.000 do treino com o **mesmo prompt de rotulagem** do Gemini (segundo rotulador);
3. roda os baselines zero-shot (prompt de baseline, uma avaliação por chamada) no teste.

Raciocínio desligado (`think: false`) e temperatura 0 em todas as chamadas.
Não recebe nenhuma chave; tudo é salvo em /kaggle/working.

In [ ]:
# Trecho comum aos notebooks que usam Ollama no Kaggle (copiado para cada notebook
# por kaggle/gerar_notebooks.py). Instala o Ollama, sobe o servidor e baixa os modelos.
import glob, os, subprocess, sys, time, json
from pathlib import Path
import requests

SRC = os.path.dirname(glob.glob("/kaggle/input/**/amostra.parquet", recursive=True)[0])
sys.path.insert(0, SRC)
os.environ["CR_DATA_PROCESSED"] = SRC
SAIDA = Path("/kaggle/working")
print("dados e código em", SRC, sorted(os.listdir(SRC)))
subprocess.run("nvidia-smi", shell=True)


def subir_ollama(modelos):
    subprocess.run("apt-get install -y -qq zstd > /dev/null 2>&1; "
                   "curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
    log = open(SAIDA / "ollama_serve.log", "w")
    subprocess.Popen(["ollama", "serve"], stdout=log, stderr=log)
    for _ in range(60):
        try:
            requests.get("http://localhost:11434/api/version", timeout=2)
            break
        except requests.RequestException:
            time.sleep(2)
    for m in modelos:
        subprocess.run(["ollama", "pull", m], check=True, stdout=subprocess.DEVNULL)
    lista = subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout
    versao = requests.get("http://localhost:11434/api/version").json()
    (SAIDA / "ollama_modelos.txt").write_text(f"{versao}\n{lista}")
    print(versao, lista)

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jsonschema"], check=True)
import pandas as pd
from rotular import rotular
from inferir import inferir

amostra = pd.read_parquet(Path(SRC) / "amostra.parquet")
ids_9b = Path(SRC, "ids_9b.txt").read_text().split()
print(len(ids_9b), amostra.set_index("review_id").loc[ids_9b, "split"].value_counts().to_dict())
subir_ollama(["qwen3.5:9b", "qwen3.5:2b"])

## 1. Segundo rotulador: Qwen3.5-9B com o prompt de rotulagem (lotes de 25)

In [ ]:
inicio = time.time()
rotular(amostra[amostra["review_id"].isin(ids_9b)], "ollama", SAIDA / "rotulos_9b.jsonl",
        tamanho_lote=25, modelo="qwen3.5:9b")
print(f"rotulagem 9B: {(time.time() - inicio) / 60:.1f} min")

## 2. Baselines zero-shot no teste (uma avaliação por chamada)

In [ ]:
teste = amostra[amostra["split"] == "teste"].sort_values("review_id")
for modelo, nome in [("qwen3.5:9b", "qwen9b"), ("qwen3.5:2b", "qwen2b")]:
    inicio = time.time()
    inferir(teste, "ollama", modelo, SAIDA / f"pred_{nome}_teste.jsonl")
    print(f"baseline {modelo}: {(time.time() - inicio) / 60:.1f} min")

In [ ]:
for arq in sorted(SAIDA.glob("*.jsonl")):
    print(arq.name, sum(1 for _ in open(arq)))